# SynFS Refactored Validation Notebook
This notebook verifies the refactored SynFS implementation.

## Section A — Quick Verification

In [9]:

import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from models.synfs_model import SynFSModel
from trainer.trainer import SynFSTrainer
from data.dataset import SimpleDataset
from omegaconf import OmegaConf

print("Imports successful ✔")

def make_quick_synthetic(n_samples=200, views_dims=[20, 30]):
    X = [np.random.randn(n_samples, d).astype(np.float32) for d in views_dims]
    y = (np.random.rand(n_samples) > 0.5).astype(int)
    return X, y

views_dims = [20, 30]
X, y = make_quick_synthetic(200, views_dims)

loader = DataLoader(SimpleDataset(X, y), batch_size=32, shuffle=True)

cfg = OmegaConf.create({
    "model": {
        "views_dims": views_dims,
        "hidden_dims": [32,48],
        "output_dim":2,
        "batch_norm":True,
        "dropout":True,
        "activation": "relu",
        "learning_rate": 1e-3,
        "s_learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "s_lam": 0.1,
        "ns_lam": 0.1,
        "ns_alpha": 0.2,
    },
    "nr_epochs": 2,
    "seed": 0,
    "device": "cpu",

})
model = SynFSModel(cfg.model).to(cfg.device)
trainer = SynFSTrainer(cfg, model)

trainer.train(loader)



Imports successful ✔
[Epoch 1] Train AUROC = 0.4706
[Epoch 2] Train AUROC = 0.5188


## Section B — Full Reproduction

In [10]:
views_dims = [250,250]

In [11]:
cfg = OmegaConf.create({
    "model": {
        "views_dims": views_dims,
        "hidden_dims": [32,32],
        "output_dim":2,
        "batch_norm":True,
        "dropout":True,
        "activation": "relu",
        "learning_rate": 1e-3,
        "s_learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "s_lam": 0.1,
        "ns_lam": 1.07,
        "ns_alpha": 0.25,
    },
    "nr_epochs": 90,
    "seed": 0,
    "device": "cpu"
})

In [12]:
# ================================
# Section B: Synthetic Dataset (Exact Match)
# ================================
from data.synfs_synthetic import (
    generate_multi_dataset,
    split_dataset,
    SimpleDataset
)
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Generate EXACT SAME data as original notebook ----
views, y, (a_gt, syn_gt) = generate_multi_dataset(
    n=20000,
    dims=views_dims,
    seed=0
)

# ---- Train/Val/Test split (IDENTICAL) ----
(tr_X_set, tr_y), (va_X_set, va_y), (te_X_set, te_y) = split_dataset(views, y)

# ---- Wrap into PyTorch loaders ----
train_data = SimpleDataset(tr_X_set, tr_y, device)
val_data   = SimpleDataset(va_X_set, va_y, device)



train_loader = DataLoader(train_data, batch_size=250, drop_last=True, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=250, drop_last=False, shuffle=False)

print("Train size:", len(train_data))
print("Val size:", len(val_data))


validate: (array([0., 1.]), array([ 9876, 10124]))
Train size: 12800
Val size: 3200


In [13]:
model = SynFSModel(cfg.model).to(cfg.device)
trainer = SynFSTrainer(cfg,model)

In [14]:
model

SynFSModel(
  (s_model): FS_predictor(
    (s_selectors): ModuleList(
      (0-1): 2 x Selector()
    )
    (s_selector_0): Selector()
    (s_selector_1): Selector()
    (shared_predictor): MLP(
      (layers): Sequential(
        (0): Sequential(
          (0): Linear(in_features=500, out_features=32, bias=True)
          (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): Dropout(p=0.5, inplace=True)
          (3): ReLU(inplace=True)
        )
        (1): Sequential(
          (0): Linear(in_features=32, out_features=32, bias=True)
          (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): Dropout(p=0.5, inplace=True)
          (3): ReLU(inplace=True)
        )
        (2): Linear(in_features=32, out_features=2, bias=True)
      )
    )
  )
  (ns_model): FS_predictor(
    (s_selectors): ModuleList(
      (0-1): 2 x Selector()
    )
    (s_selector_0): Selector()
    (s_selector_1): Sel

In [15]:
trainer.model.s_model.s_selector_0.mu

Parameter containing:
tensor([ 1.1960e-02, -1.1783e-02, -8.4935e-03, -2.2973e-04, -1.5367e-02,
        -1.7270e-02,  8.5364e-03, -3.3574e-02, -7.5663e-03, -3.6751e-03,
        -3.0986e-02, -4.6813e-03,  7.7708e-03, -5.1259e-03, -4.9244e-03,
         5.2639e-04, -1.5351e-02, -8.8054e-03, -2.6698e-03,  6.8465e-03,
        -9.6327e-03,  8.2985e-03, -9.8270e-04,  1.2081e-02,  4.1048e-03,
        -7.3441e-03,  4.4261e-03, -9.9240e-03, -3.0745e-03,  1.3611e-02,
         8.0581e-04, -5.9253e-03,  5.4001e-03,  3.4864e-03,  1.0237e-02,
        -1.5223e-03,  5.8990e-03, -7.3677e-03, -7.0586e-03, -8.7051e-03,
        -6.1929e-03,  1.1526e-02, -8.1401e-03,  3.3111e-03,  1.4975e-02,
        -7.0552e-03, -2.1048e-02,  2.5259e-03,  2.1557e-03, -2.0643e-02,
        -9.7619e-03, -6.4225e-03, -1.8208e-03,  5.2549e-03, -1.1936e-03,
         2.9979e-03,  7.4511e-03, -4.1938e-04,  8.2654e-03, -3.7886e-03,
         4.6518e-03, -6.7333e-03, -9.0467e-03,  1.1824e-02, -8.0355e-03,
        -2.2902e-03,  1.1424e

In [17]:





history = {"train_loss": [], "val_auroc": []}
trainer.set_X_mean_set(train_loader)

for epoch in range(cfg.nr_epochs):
    if epoch ==0:
        print('0 epoch predicted synergistic features')
        print(np.where(np.concatenate([g.cpu().numpy() for g in model.get_gates(model.s_model)])>0.7)[0])
        print('predicted non-synergistic features')
        print(np.where(np.concatenate([g.cpu().numpy() for g in model.get_gates(model.ns_model)])>0.7)[0])

    train_metrics = trainer.train_epoch(train_loader)
    val_metrics   = trainer.validate_epoch(val_loader)

    history["train_loss"].append(train_metrics["loss"])
    history["val_auroc"].append(val_metrics["auroc"])


    if epoch % 30 == 0:
        print("======Epoch", epoch, "Train Loss:", train_metrics['loss'], "Val AUROC:", val_metrics['auroc'], "======")
        

        print('predicted synergistic features')
        print(np.where(np.concatenate([g.cpu().numpy() for g in model.get_gates(model.s_model)])>0.7)[0])
        print('predicted non-synergistic features')
        print(np.where(np.concatenate([g.cpu().numpy() for g in model.get_gates(model.ns_model)])>0.7)[0])


0 epoch predicted synergistic features
[  0 117 176 251 276 358 449]
predicted non-synergistic features
[  2 253]
======Epoch 0 Train Loss: 0.9787738440083522 Val AUROC: 0.7772408088887973 ======
predicted synergistic features
[  0  40  65 176 251 276 358 449 459]
predicted non-synergistic features
[  2 253]


KeyboardInterrupt: 